# 🎧 AUDIO 

Converts your `my_book.epub` into an expressive `.wav` audiobook.

> ⚠️ **GPU required!** Go to `Runtime → Change runtime type → T4 GPU`.

## 1. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/dhruv0rathore/AUDIO.git
%cd AUDIO

In [ ]:
# Install all deps EXCEPT torch/torchaudio/torchvision — Colab already has compatible versions pre-installed.
# Overwriting them causes CUDA version mismatches (e.g. libcudart.so.13 not found).
!pip install -q PyMuPDF EbookLib beautifulsoup4 nltk transformers datasets accelerate scipy pydub pdfplumber mobi
!pip install -q git+https://github.com/suno-ai/bark.git

In [ ]:
!python download_nltk.py

## 2. Upload your `my_book.epub`

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
uploaded_name = list(uploaded.keys())[0]

# main.py expects the file to be named my_book.epub
if uploaded_name != "my_book.epub":
    os.rename(uploaded_name, "my_book.epub")
    print(f"Renamed '{uploaded_name}' → 'my_book.epub'")

print(f"✅ Ready: my_book.epub ({os.path.getsize('my_book.epub')} bytes)")

## 3. Run the Pipeline

We need a small wrapper because Colab's PyTorch 2.6+ defaults `torch.load(weights_only=True)`, which crashes Bark's model loading. The wrapper patches this before running `main.py`.

In [ ]:
%%writefile run_colab.py
# Patch torch.load for PyTorch 2.6+ compatibility with Bark
# Bark's checkpoints use numpy globals that aren't in the safe list,
# so we need weights_only=False.
import torch
_orig_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _patched_load

# Run main.py
exec(open('main.py').read())

In [ ]:
!python run_colab.py

## 4. Play & Download

In [ ]:
import IPython.display as ipd
from google.colab import files
import os

OUTPUT_FILE = "final_prompted_audiobook.wav"

if not os.path.exists(OUTPUT_FILE):
    print(f"Current directory: {os.getcwd()}")
    print(f"Files here: {os.listdir('.')}")
    raise FileNotFoundError(
        f"'{OUTPUT_FILE}' not found. Make sure step 3 completed successfully."
    )

print(f"🔊 Playing generated audiobook ({os.path.getsize(OUTPUT_FILE)} bytes):")
ipd.display(ipd.Audio(filename=OUTPUT_FILE))

print("\n📥 Downloading .wav file...")
files.download(OUTPUT_FILE)